<a href="https://colab.research.google.com/github/snowa0719/Oral-Pill-Objest-Detection_TEAM_04/blob/main/base_line_Faster_R_CNN%2BEfficientNet_B3_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> (중요) 여러분 구글 드라이브에 최소 7GB 이상은 확보되어 있어야 합니다!

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import unicodedata  # 0번 섹션에 추가 필요

# 📌 경로 설정 (제공해주신 경로 반영)
def normalize_path(path):
    # 1. unicodedata.normalize('NFC', path): 경로 문자열을 NFC 방식으로 통일
    # 2. .strip(): 앞뒤에 붙은 불필요한 공백 제거
    return unicodedata.normalize('NFC', path).strip()

> (주의) 아래 코드는 처음 딱 한 번만!

In [ ]:
# import zipfile
# import os
# import shutil
# import time

# # 1. 경로 설정
# dataset_zip = normalize_path("/content/drive/MyDrive/data/초급_프로젝트/초급_프로젝트_수강생_배포용.zip")
# extract_path = normalize_path("/content/drive/MyDrive/data/초급_프로젝트/dataset/")

# os.makedirs(extract_path, exist_ok=True)

# # 2. 메인 압축파일 해제
# print(f"📦 메인 데이터셋 해제 중: {os.path.basename(dataset_zip)}")
# with zipfile.ZipFile(dataset_zip, 'r') as zip_ref:
#     zip_ref.extractall(extract_path)

# # 메인 압축파일 삭제 (요청사항)
# os.remove(dataset_zip)
# print("🗑️ 메인 압축파일 삭제 완료.")

# # 3. 내부 이미지 압축파일 통합 해제 로직
# print("\n🚀 이미지 폴더 통합 및 내부 압축 해제 시작...")
# for file in os.listdir(extract_path):
#     if file.endswith(".zip"):
#         file_path = os.path.join(extract_path, file)

#         # 이름 기반 대상 폴더 결정 (train/test 통합)
#         if 'train' in file.lower():
#             target_folder_name = "train_images"
#         elif 'test' in file.lower():
#             target_folder_name = "test_images"
#         else:
#             target_folder_name = file.replace(".zip", "")

#         target_subfolder = os.path.join(extract_path, target_folder_name)
#         os.makedirs(target_subfolder, exist_ok=True)

#         print(f"📂 {file} -> {target_folder_name} 통합 중...")

#         with zipfile.ZipFile(file_path, 'r') as zip_ref:
#             for member in zip_ref.infolist():
#                 if not member.is_dir():
#                     # 내부 경로 구조를 무시하고 파일명만 추출하여 저장
#                     filename = os.path.basename(member.filename)
#                     if filename:
#                         target_file_path = os.path.join(target_subfolder, filename)
#                         with zip_ref.open(member) as source, open(target_file_path, "wb") as target:
#                             shutil.copyfileobj(source, target)

#         # [수정 포인트] 삭제 전 잠시 대기 후 강제 삭제 시도
#         try:
#             time.sleep(0.5)
#             if os.path.exists(file_path):
#                 os.remove(file_path)
#                 print(f"🗑️ 삭제 성공: {file}")
#         except Exception as e:
#             print(f"❌ {file} 삭제 실패: {e}")

# print("\n✨ 모든 작업이 완료되었습니다!")
# print(f"📁 최종 데이터셋 구성: {os.listdir(extract_path)}")

- 구글 드라이브 휴지통 비우기

In [ ]:
# from google.colab import auth
# from googleapiclient.discovery import build

# # 1. 구글 드라이브 인증
# auth.authenticate_user()
# drive_service = build('drive', 'v3')

# # 2. 휴지통 완전히 비우기 함수
# def empty_trash():
#     try:
#         drive_service.files().emptyTrash().execute()
#         print("✅ 구글 드라이브 휴지통이 완전히 비워졌습니다.")
#     except Exception as e:
#         print(f"❌ 휴지통 비우기 실패: {e}")

# # 실행
# empty_trash()

> 압축 해제한 파일들의 반영 시간이 걸릴 수 있으므로, 커널 재시작 해주기!

#### `normalize_path`는?

`normalize_path`는 파일 경로에 포함된 **한글(유니코드) 처리 방식**을 통일하여, 경로를 찾지 못하는 에러를 방지하기 위한 함수입니다.

특히 **Google Colab**이나 **Mac, Windows** 사이에서 데이터를 주고받을 때 한글 폴더명이 깨져서 발생하는 `File Not Found` 에러를 잡는 데 필수적입니다.

<br>

##### 왜 사용하나요? (NFC vs NFD)

한글을 컴퓨터가 인식하는 방식은 크게 두 가지입니다.

* **NFC (Windows 스타일):** '강'을 '강'이라는 하나의 글자로 저장합니다.
* **NFD (Mac/Unix 스타일):** '강'을 'ㄱ', 'ㅏ', 'ㅇ'으로 쪼개서 저장합니다.

사람 눈에는 똑같이 "초급 프로젝트"라고 보이지만, 컴퓨터 입장에서는 글자 조합 방식이 다르면 **완전히 다른 경로**로 인식합니다. `normalize_path` 는 이를 **NFC(표준 방식)** 로 강제 통일해주는 역할을 합니다.

In [ ]:
!pip install -q optuna

In [ ]:
############################################################
# 0. 라이브러리 임포트 & 경로 설정
############################################################
import os
import json
import pandas as pd
from PIL import Image
import unicodedata
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim
import torchvision
import torchvision.transforms as T
from collections import OrderedDict

from torchvision.models import efficientnet_b3, EfficientNet_B3_Weights
from torchvision.models.feature_extraction import create_feature_extractor
from torchvision.models.detection import FasterRCNN
from torchvision.models.detection.rpn import AnchorGenerator
from torchvision.ops import FeaturePyramidNetwork, MultiScaleRoIAlign
from torchvision.ops.feature_pyramid_network import LastLevelMaxPool


# 📌 경로 설정 (제공해주신 경로 반영)
def normalize_path(path):
    # 1. unicodedata.normalize('NFC', path): 경로 문자열을 NFC 방식으로 통일
    # 2. .strip(): 앞뒤에 붙은 불필요한 공백 제거
    return unicodedata.normalize('NFC', path).strip()

# 경로는 환경에 맞게 수정
# train_images, test_images
extract_path = normalize_path("/content/drive/MyDrive/data/초급_프로젝트/dataset/") # 압축을 풀 폴더

TRAIN_JSON_PATH = os.path.join(extract_path, "merged_annotations_train_final.json")
TEST_JSON_PATH = os.path.join(extract_path, "merged_annotations_test_final.json")
TRAIN_IMG_DIR = os.path.join(extract_path, "train_images")
TEST_IMG_DIR  = os.path.join(extract_path, "test_images")

# merged_annotation json 경로
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [ ]:

# 코랩 추가 데이터 파일 압축 해제

import zipfile

# 1. 업로드된 파일명 확인 (코랩 로컬 /content/ 기준)
# 혹시 파일명 뒤에 (1) 등이 붙었는지 확인하기 위해 리스트를 뽑습니다.
current_files = os.listdir('/content/')
print("현재 업로드된 파일 목록:", current_files)

# 2. 압축 해제 설정
zip_targets = {
    "images": "/content/new_images_1000.zip",
    "labels": "/content/TL_81_단일.zip"
}

extract_dirs = {
    "images": "/content/new_images_1000",
    "labels": "/content/TL_81_단일"
}

for key, zip_path in zip_targets.items():
    if os.path.exists(zip_path):
        target_dir = extract_dirs[key]
        if not os.path.exists(target_dir):
            print(f"📦 {zip_path} 압축 해제 중...")
            with zipfile.ZipFile(zip_path, 'r') as zip_ref:
                zip_ref.extractall(target_dir)
            print(f"✅ {target_dir} 생성 완료!")
        else:
            print(f"📂 이미 압축이 풀려있습니다: {target_dir}")
    else:
        print(f"❌ 에러: {zip_path} 파일이 /content/ 안에 없습니다. 업로드를 완료해 주세요!")


# 4. 이후 코드에서 사용할 경로 변수 설정
EXTRA_IMG_DIR = "/content/new_images_1000"
EXTRA_JSON_DIR = "/content/TL_81_단일"

현재 업로드된 파일 목록: ['.config', 'optuna_gt_temp.json', 'drive', 'TL_81_단일.zip', 'new_images_1000', 'temp_optuna.json', 'new_images_1000.zip', 'TL_81_단일', 'sample_data']
📂 이미 압축이 풀려있습니다: /content/new_images_1000
📂 이미 압축이 풀려있습니다: /content/TL_81_단일


In [ ]:
############################################################
# 1. 원본 데이터 로드 + 추가 데이터(1000장) 병합
############################################################

def build_df_from_merged_json(json_path, img_dir):
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    id_to_fname = {img["id"]: img["file_name"] for img in data["images"]}
    records = []
    for ann in data["annotations"]:
        img_id_coco = ann["image_id"]
        file_name = id_to_fname.get(img_id_coco)
        if file_name is None: continue
        img_path = os.path.join(img_dir, file_name)
        if not os.path.exists(img_path): continue
        x, y, w, h = ann["bbox"]
        records.append({
            "image_path": img_path,
            "image_id": os.path.splitext(file_name)[0],
            "category_id": int(ann["category_id"]),
            "bbox_x": float(x), "bbox_y": float(y), "bbox_w": float(w), "bbox_h": float(h),
        })
    return pd.DataFrame(records)

# 1-1. 기본 데이터 로드
df = build_df_from_merged_json(TRAIN_JSON_PATH, TRAIN_IMG_DIR)

# 1-2. 추가 이미지 및 라벨 병합 로직 (YOLO 코드에서 가져옴)
EXTRA_IMG_DIR = "/content/new_images_1000/new_images"
EXTRA_JSON_DIR = "/content/TL_81_단일/TL_81_단일"

extra_records = []
existing_ids = set(df["image_id"].tolist())

if os.path.exists(EXTRA_IMG_DIR):
    for img_fname in os.listdir(EXTRA_IMG_DIR):
        if not img_fname.lower().endswith(('.jpg', '.jpeg', '.png')): continue

        stem = os.path.splitext(img_fname)[0]
        if stem in existing_ids: continue # 중복 방지

        # 추가 데이터의 JSON 경로 생성 (YOLO 로직과 동일)
        kid = stem.split('_')[0]
        json_path = os.path.join(EXTRA_JSON_DIR, kid + "_json", stem + ".json")

        if os.path.exists(json_path):
            with open(json_path, "r", encoding="utf-8") as f:
                extra_data = json.load(f)

            for ann in extra_data["annotations"]:
                x, y, w, h = ann["bbox"]
                extra_records.append({
                    "image_path": os.path.join(EXTRA_IMG_DIR, img_fname),
                    "image_id": stem,
                    "category_id": int(ann["category_id"]),
                    "bbox_x": float(x), "bbox_y": float(y), "bbox_w": float(w), "bbox_h": float(h),
                })

    extra_df = pd.DataFrame(extra_records)
    df = pd.concat([df, extra_df], ignore_index=True)
    print(f"✅ 추가 데이터 {len(extra_records)}개 병합 완료. 총 객체 수: {len(df)}")
else:
    print("⚠️ 추가 이미지 경로를 찾을 수 없습니다. 기본 데이터만 사용합니다.")

✅ 추가 데이터 482개 병합 완료. 총 객체 수: 5008


In [ ]:
############################################################
# 2. category_id 매핑 (겉으로는 안 바꾸고, 모델 내부에서만 사용)
############################################################

# 원본 category_id 집합
unique_cats = sorted(df["category_id"].unique())
print("고유 category_id 개수:", len(unique_cats))

# 내부용: 모델에 넣을 label (1 ~ num_classes-1), 0은 background
orig2model = {cid: i + 1 for i, cid in enumerate(unique_cats)}   # 원본 → 모델용
model2orig = {v: k for k, v in orig2model.items()}               # 모델용 → 원본

num_classes = len(unique_cats) + 1  # background 포함
print("num_classes (background 포함):", num_classes)

고유 category_id 개수: 74
num_classes (background 포함): 75


In [ ]:
# 모든 고유 문자열 ID를 리스트로 뽑고, 각각에 고유 숫자(0, 1, 2...)를 부여합니다.
all_unique_names = df['image_id'].unique().tolist()
name2int_id = {name: i for i, name in enumerate(all_unique_names)}

print(f"총 {len(name2int_id)}개의 고유 이미지가 숫자로 매핑되었습니다.")
# 예: 'K-001900...' -> 0

총 1971개의 고유 이미지가 숫자로 매핑되었습니다.


In [ ]:
########### [최종] 정답지 JSON 업데이트 및 ID 동기화 #############

# 1. 기존 정답지 로드 (YOLO와 비교할 원본 시험지)
with open(TEST_JSON_PATH, 'r') as f:
    coco_data = json.load(f)

new_name2int_id = {}
new_images = []
old_to_new_id = {}

# 2. JSON에 있는 이미지(843개)를 기준으로 0, 1, 2... 새 번호표 발행
for i, img in enumerate(coco_data['images']):
    file_name = img.get('file_name', '')
    old_id = img.get('id', '')
    pure_name = file_name.split('.')[0] # 확장자 제거 버전

    new_id = i # 0부터 시작하는 새로운 정수 ID
    img['id'] = new_id
    new_images.append(img)

    # [핵심] Dataset(df)에서 어떤 이름으로 불러도 이 번호(new_id)를 찾을 수 있게 등록
    new_name2int_id[pure_name] = new_id  # 확장자 없는 경우 대비
    new_name2int_id[file_name] = new_id  # 확장자 있는 경우 대비

    # 기존 JSON 내의 annotation ID들을 바뀐 ID로 연결하기 위해 기록
    old_to_new_id[old_id] = new_id

# 3. 정답(Annotations)의 image_id도 새로운 숫자 ID로 일괄 업데이트
new_annotations = []
for ann in coco_data['annotations']:
    old_img_id = ann['image_id']
    if old_img_id in old_to_new_id:
        ann['image_id'] = old_to_new_id[old_img_id]
        new_annotations.append(ann)

coco_data['images'] = new_images
coco_data['annotations'] = new_annotations

# 4. Optuna 전용 임시 정답지 저장
with open("optuna_gt_temp.json", "w") as f:
    json.dump(coco_data, f)

# 5. 전역 변수 name2int_id 갱신 (Dataset 클래스가 이 변수를 참조함)
name2int_id = new_name2int_id

print(f"✅ 동기화 완료! 총 {len(new_images)}개 이미지에 대한 번호표(0~{len(new_images)-1})가 생성되었습니다.")

✅ 동기화 완료! 총 843개 이미지에 대한 번호표(0~842)가 생성되었습니다.


In [ ]:
############################################################
# 3. Dataset 정의
############################################################

from torchvision import transforms as T

class OralDrugDataset(Dataset):
    def __init__(self, df, orig2model, transforms=None):
        self.df = df.reset_index(drop=True)
        self.orig2model = orig2model
        self.transforms = transforms
        self.image_ids = self.df["image_id"].unique().tolist()

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        image_id_name = self.image_ids[idx] # 현재 인덱스의 문자열 ID
        df_img = self.df[self.df["image_id"] == image_id_name]

        img_path = df_img["image_path"].iloc[0]
        image = Image.open(img_path).convert("RGB")

        # 1) 원본 이미지 크기 기록
        orig_w, orig_h = image.size

        # 2) 모델 입력 크기 설정 (EfficientNet-B3 권장 크기)
        target_size = 640

        # 3) 이미지 리사이즈
        image = image.resize((target_size, target_size))

        boxes = []
        labels = []

        # [추가] 모델이 허용하는 최대 라벨 번호 (배경 0을 제외한 실제 클래스 개수)
        max_label_idx = len(self.orig2model)

        for _, row in df_img.iterrows():
            x, y, w, h = row["bbox_x"], row["bbox_y"], row["bbox_w"], row["bbox_h"]
            x1 = max(0, min(x * (target_size / orig_w), target_size))
            y1 = max(0, min(y * (target_size / orig_h), target_size))
            x2 = max(0, min((x + w) * (target_size / orig_w), target_size))
            y2 = max(0, min((y + h) * (target_size / orig_h), target_size))

            if (x2 > x1) and (y2 > y1):
                boxes.append([x1, y1, x2, y2])

                # [수정 1] 라벨 번호가 범위를 넘지 않게 고정
                # 만약 번호가 76이 나오면 강제로 max_label_idx(75)로 맞춤
                l = self.orig2model[int(row["category_id"])] + 1
                labels.append(min(l, max_label_idx))

        # 6) 만약 박스가 하나도 없다면
        if len(boxes) == 0:
            boxes = torch.tensor([[0.0, 0.0, 1.0, 1.0]], dtype=torch.float32)
            # [수정 2] 기본 라벨도 배경(0)이 아닌 1번 클래스로 안전하게 설정
            labels = torch.tensor([1], dtype=torch.int64)
        else:
            boxes = torch.tensor(boxes, dtype=torch.float32)
            labels = torch.tensor(labels, dtype=torch.int64)

        # name2int_id에서 숫자 ID를 안전하게 가져옵니다.
        # 1순위: 파일명 그대로 찾기 / 2순위: 확장자 떼고 찾기 / 3순위: 없으면 -1
        try:
            int_id = name2int_id.get(image_id_name, name2int_id.get(image_id_name.split('.')[0], -1))
        except:
            int_id = -1

        # COCOeval 호환을 위해 1차원 정수 텐서로 변환
        # 정답지(JSON)에 없는 이미지라면 -1이 들어갑니다.
        image_id_tensor = torch.tensor([int_id], dtype=torch.int64)

        target = {
            "boxes": boxes,
            "labels": labels,
            "image_id": image_id_tensor,
        }

        # 7) 텐서 변환 (ToTensor 등)
        if self.transforms is not None:
            image = self.transforms(image)
        else:
            # transforms가 없을 경우 최소한 텐서로는 바꿔야 함
            image = T.ToTensor()(image)

        return image, target

############################################################
# 4. Transform, Dataset, DataLoader 구성
############################################################

train_transforms = T.Compose([
    # 필요하다면 여기서 RandomHorizontalFlip 등 augmentation 추가
    T.ToTensor(),  # PIL 이미지를 [0,1] float tensor로 변환
])

dataset = OralDrugDataset(df, orig2model=orig2model, transforms=train_transforms)

# 간단히 train/val split (예: 90% train, 10% val)
indices = torch.randperm(len(dataset)).tolist()
split = int(0.9 * len(indices))
train_indices = indices[:split]
val_indices   = indices[split:]

train_dataset = torch.utils.data.Subset(dataset, train_indices)
val_dataset   = torch.utils.data.Subset(dataset, val_indices)

def collate_fn(batch):
    # detection 모델용 collate_fn: 리스트의 튜플을 튜플의 리스트로
    return tuple(zip(*batch))

train_loader = DataLoader(
    train_dataset,
    batch_size=2,              # GPU 메모리에 맞게 조정
    shuffle=True,
    collate_fn=collate_fn,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=2,
    shuffle=False,
    collate_fn=collate_fn,
)

print("train steps:", len(train_loader), "val steps:", len(val_loader))



train steps: 887 val steps: 99


In [ ]:
from torch.utils.data import DataLoader

# 위에서 정의한 train_dataset, val_dataset을 Optuna용 로더에 연결
optuna_train_loader = DataLoader(
    train_dataset,
    batch_size=2,        # 메모리 안전을 위해 2 유지
    shuffle=True,
    collate_fn=collate_fn
)

optuna_val_loader = DataLoader(
    val_dataset,
    batch_size=2,
    shuffle=False,
    collate_fn=collate_fn
)

print(f"✅ Optuna용 로더 준비 완료! (Train: {len(optuna_train_loader)} batches)")

✅ Optuna용 로더 준비 완료! (Train: 887 batches)


In [ ]:
import torch

# 1. GPU 장치 설정
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

# 2. 데이터 로더가 이미 선언되어 있는지 확인 (만약 이름이 다르면 여기서 연결)
# 만약 앞에서 만든 로더 이름이 train_loader, val_loader라면 아래처럼 연결해줍니다.
try:
    optuna_train_loader = optuna_train_loader
    optuna_val_loader = optuna_val_loader
except NameError:
    # 혹시 로더 이름이 다를 경우를 대비해 기존에 정의된 로더를 할당하세요.
    # 예: optuna_train_loader = train_loader
    print("⚠️ 'optuna_train_loader'를 찾을 수 없습니다. 위쪽에서 데이터 로더 셀을 먼저 실행해주세요!")

print(f"✅ 준비 완료! 장치: {device}")

✅ 준비 완료! 장치: cuda


In [ ]:
############################################################
# [추가] EfficientNet-B3 backbone + FPN 정의
############################################################

class EfficientNetB3BackboneWithFPN(nn.Module):
    def __init__(self, pretrained=True, out_channels=256):
        super().__init__()

        weights = EfficientNet_B3_Weights.DEFAULT if pretrained else None
        base_model = efficientnet_b3(weights=weights)

        # EfficientNet-B3에서 중간 feature map 추출
        # 보통 stride 4, 8, 16, 32 수준의 feature를 사용
        return_nodes = {
            "features.2": "0",
            "features.3": "1",
            "features.5": "2",
            "features.7": "3",
        }

        self.body = create_feature_extractor(base_model, return_nodes=return_nodes)

        # 각 feature map의 channel 수 자동 추론(입력 크기를 640으로 맞춤)
        with torch.no_grad():
            dummy = torch.zeros(1, 3, 640, 640)
            feats = self.body(dummy)
            in_channels_list = [feat.shape[1] for feat in feats.values()]

        self.fpn = FeaturePyramidNetwork(
            in_channels_list=in_channels_list,
            out_channels=out_channels,
            extra_blocks=LastLevelMaxPool()
        )

        # FasterRCNN에서 필요
        self.out_channels = out_channels

    def forward(self, x):
        feats = self.body(x)   # dict 형태
        feats = OrderedDict((k, v) for k, v in feats.items())
        feats = self.fpn(feats)
        return feats


def get_model(num_classes):
    backbone = EfficientNetB3BackboneWithFPN(pretrained=True, out_channels=256)

    # 일반적인 anchor 설정
    # 만약 객체가 작으면 (16, 32, 64, 128, 256)로 바꿔도 됨
    anchor_generator = AnchorGenerator(
        sizes=((32,), (64,), (128,), (256,), (512,)),
        aspect_ratios=((0.5, 1.0, 2.0),) * 5
    )

    # FPN feature map 중 어떤 레벨을 ROI pooling에 쓸지 지정
    box_roi_pool = MultiScaleRoIAlign(
        featmap_names=["0", "1", "2", "3"],
        output_size=7,
        sampling_ratio=2
    )

    actual_num_classes = num_classes + 1 # 배경(0)을 위해 1을 더함

    model = FasterRCNN(
        backbone=backbone,
        num_classes=actual_num_classes,
        rpn_anchor_generator=anchor_generator,
        box_roi_pool=box_roi_pool,
        min_size=640, # Dataset의 target_size와 동일하게
        max_size=640
    )

    return model

print("✅ EfficientNet-B3 모델 생성 함수(get_model) 정의 완료!")


✅ EfficientNet-B3 모델 생성 함수(get_model) 정의 완료!


In [ ]:
############################################################
# 4.5 옵튜나 학습 (mAP 기반 최적화 - 30% Subset)
############################################################

import optuna
import torch.optim as optim
from torch.utils.data import Subset
import numpy as np
from tqdm.auto import tqdm
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
import json

# 0. 결과 일관성을 위한 시드 고정
np.random.seed(42)

# 0-1. 옵튜나 전용 데이터 로더 (30% Subset)
indices = np.arange(len(train_dataset))
np.random.shuffle(indices)
subset_size = int(0.3 * len(train_dataset))
optuna_subset_indices = indices[:subset_size]
optuna_train_subset = Subset(train_dataset, optuna_subset_indices)

optuna_train_loader = DataLoader(
    optuna_train_subset,
    batch_size=train_loader.batch_size,
    shuffle=True,
    collate_fn=collate_fn
)

def set_bn_eval(m):
    if isinstance(m, (nn.BatchNorm2d, nn.SyncBatchNorm)):
        m.eval()


In [ ]:

# [mAP 최적화] mAP 계산을 위한 보조 함수

def evaluate_map(model, data_loader, device):
    model.eval()
    results = []
    with torch.no_grad():
        for images, targets in data_loader:
            images = [img.to(device) for img in images]
            outputs = model(images)
            for i, output in enumerate(outputs):
                img_id = targets[i]["image_id"].item()
                if img_id == -1: continue # 정답지에 없는 이미지는 채점 제외

                boxes = output["boxes"].cpu().numpy()
                scores = output["scores"].cpu().numpy()
                labels = output["labels"].cpu().numpy()
                for box, score, label in zip(boxes, scores, labels):
                    # 모델이 내뱉은 label에서 1을 빼야 원래 category_id를 찾을 수 있음
                    orig_idx = int(label) - 1

                    # 안전장치: 혹시라도 인덱스가 음수가 되거나 범위를 벗어나면 무시
                    if orig_idx in model2orig:
                        results.append({
                            "image_id": img_id,
                            "category_id": int(model2orig[orig_idx]),
                            "bbox": [
                                float(box[0]),
                                float(box[1]),
                                float(box[2] - box[0]),
                                float(box[3] - box[1])
                            ],
                            "score": float(score)
                        })



    if not results: return 0.0
    with open("temp_optuna.json", "w") as f: json.dump(results, f)

    try:
        from pycocotools.coco import COCO
        from pycocotools.cocoeval import COCOeval
        coco_gt = COCO("optuna_gt_temp.json")
        coco_dt = coco_gt.loadRes("temp_optuna.json")
        coco_eval = COCOeval(coco_gt, coco_dt, "bbox")

        # [핵심] 정답지에 있는 모든 ID를 대상으로 평가하겠다고 선언
        coco_eval.params.imgIds = coco_gt.getImgIds()

        coco_eval.evaluate()
        coco_eval.accumulate()
        coco_eval.summarize()
        return float(coco_eval.stats[0])
    except:
        return 0.0


In [ ]:
def objective(trial):
    # 하이퍼파라미터 샘플링
    lr_backbone = trial.suggest_float("lr_backbone", 1e-6, 1e-4, log=True)
    lr_head = trial.suggest_float("lr_head", 1e-5, 1e-3, log=True)
    weight_decay = trial.suggest_float("weight_decay", 1e-5, 1e-3, log=True)

    # 모델 초기화 (매 Trial마다 새로 시작)
    model = get_model(num_classes=len(model2orig)).to(device)

    params = [
        {"params": model.backbone.parameters(), "lr": lr_backbone},
        {"params": model.roi_heads.parameters(), "lr": lr_head},
    ]
    optimizer = torch.optim.AdamW(params, weight_decay=weight_decay)
    scaler = torch.amp.GradScaler()

    for epoch in range(5): # 시간 관계상 5회로 조절
        model.train()
        total_loss = 0
        for images, targets in optuna_train_loader:
            images = [img.to(device) for img in images]
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

            with torch.amp.autocast('cuda'):
                loss_dict = model(images, targets)
                losses = sum(loss for loss in loss_dict.values())

            optimizer.zero_grad()
            scaler.scale(losses).backward()
            scaler.step(optimizer)
            scaler.update()

            total_loss += losses.item()

        avg_loss = total_loss / len(optuna_train_loader)
        # ---------------- [이 부분이 중요!] ----------------
        print(f"   [Trial {trial.number}] Epoch {epoch+1} - Avg Loss: {avg_loss:.4f}")
        # --------------------------------------------------

    # 최종 평가 (mAP 계산)
    current_map = evaluate_map(model, optuna_val_loader, device)
    print(f"👉 Trial {trial.number} 최종 mAP: {current_map:.4f}")

    return current_map


# Study 생성 (mAP는 높을수록 좋으므로 maximize)
study = optuna.create_study(
    direction="maximize",
    pruner=optuna.pruners.MedianPruner()
)


[I 2026-03-27 17:15:39,914] A new study created in memory with name: no-name-0513ec97-7f3e-4fb6-96ea-c1f9572bf17b


In [ ]:
# 최적화 시작
print(f"🔥 하이퍼파라미터 최적화 시작 (최대 10회 시도)...")

study.optimize(
    objective,
    n_trials=10,      # 시도 횟수 (시간에 맞춰 조절하세요)
    timeout=3600,     # 최대 1시간 동안만 실행 (코랩 세션 방어용)
    n_jobs=1          # GPU 학습이므로 반드시 1로 설정
)

# 최적의 결과 출력
print("\n" + "="*30)
print("✨ 최적화 완료! ✨")
print(f"🥇 최고 mAP Score: {study.best_value:.4f}")
print("📌 최적의 파라미터:")
for key, value in study.best_params.items():
    print(f"   - {key}: {value}")
print("="*30)

🔥 하이퍼파라미터 최적화 시작 (최대 10회 시도)...
   [Trial 0] Epoch 1 - Avg Loss: 0.8410
   [Trial 0] Epoch 2 - Avg Loss: 0.5058
   [Trial 0] Epoch 3 - Avg Loss: 0.4153
   [Trial 0] Epoch 4 - Avg Loss: 0.3661
   [Trial 0] Epoch 5 - Avg Loss: 0.3374
loading annotations into memory...
Done (t=0.02s)
creating index...
index created!
Loading and preparing results...
DONE (t=0.01s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=1.03s).
Accumulating evaluation results...


[I 2026-03-27 17:23:25,401] Trial 0 finished with value: 0.0 and parameters: {'lr_backbone': 4.237536701489191e-06, 'lr_head': 0.00016697275785920753, 'weight_decay': 3.7542727565517023e-05}. Best is trial 0 with value: 0.0.


DONE (t=0.36s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.000
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.000
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.000
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = -1.000
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = -1.000
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.000
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.000
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.000
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.000
 Average Recall     (AR) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = -1.000
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = -1.000
 Average Recall     (AR) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.000
👉 Trial 0 최종 mAP: 0.

[I 2026-03-27 17:30:42,628] Trial 1 finished with value: 0.0 and parameters: {'lr_backbone': 5.831207756477942e-05, 'lr_head': 7.027468043988996e-05, 'weight_decay': 8.120129833057443e-05}. Best is trial 0 with value: 0.0.


DONE (t=0.47s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.000
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.000
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.000
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = -1.000
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = -1.000
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.000
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.000
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.000
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.000
 Average Recall     (AR) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = -1.000
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = -1.000
 Average Recall     (AR) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.000
👉 Trial 1 최종 mAP: 0.

...

DONE (t=0.32s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.000
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.000
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.000
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = -1.000
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = -1.000
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.000
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.000
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.000
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.000
 Average Recall     (AR) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = -1.000
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = -1.000
 Average Recall     (AR) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.000
👉 Trial 6 최종 mAP: 0.

[I 2026-03-27 18:13:49,435] Trial 7 finished with value: 0.0 and parameters: {'lr_backbone': 1.4980133923698088e-06, 'lr_head': 0.0005563029204585841, 'weight_decay': 1.9189107181143174e-05}. Best is trial 0 with value: 0.0.


DONE (t=0.32s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.000
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.000
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.000
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = -1.000
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = -1.000
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.000
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.000
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.000
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.000
 Average Recall     (AR) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = -1.000
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = -1.000
 Average Recall     (AR) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.000
👉 Trial 7 최종 mAP: 0.

[I 2026-03-27 18:20:56,815] Trial 8 finished with value: 0.0 and parameters: {'lr_backbone': 4.097311365804743e-06, 'lr_head': 1.5003688720475898e-05, 'weight_decay': 1.6007184653686003e-05}. Best is trial 0 with value: 0.0.


DONE (t=0.54s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.000
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.000
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.000
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = -1.000
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = -1.000
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.000
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.000
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.000
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.000
 Average Recall     (AR) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = -1.000
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = -1.000
 Average Recall     (AR) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.000
👉 Trial 8 최종 mAP: 0.

In [16]:
# DEVICE 변수 통일
# 앞서 'device'라는 소문자 변수를 썼다면, 여기서 'DEVICE'로 연결해줍니다.
if 'device' in globals():
    DEVICE = device
else:
    DEVICE = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

# num_classes 정의 (배경 포함 여부 확인)
# 1. model2orig는 실제 약 종류의 개수입니다.
# 2. Faster R-CNN은 배경(0번)이 필요하므로 num_classes는 '실제 종류 + 1'이 되어야 합니다.
if 'model2orig' in globals():
    num_classes = len(model2orig)
    # [중요] get_model 함수 내부에서 이미 +1을 하도록 짰다면 그대로 num_classes를 넣고,
    # 함수 내부에서 안 했다면 여기서 num_classes + 1을 해서 넣어야 합니다.
else:
    print("⚠️ 'model2orig' 변수가 없습니다. 위쪽 정의 셀을 먼저 실행하세요!")

# 아까 정의한 Optuna용 함수 이름을 그대로 사용합니다.
# model = get_fasterrcnn_efficientnet_b3(num_classes) -> X
# model = get_model(num_classes) -> O
try:
    model = get_model(num_classes)
    model.to(DEVICE)
    print(f"✅ 모델 로드 완료! (클래스 수: {num_classes + 1}, 장치: {DEVICE})")
except NameError:
    print("⚠️ 'get_model' 함수가 정의되지 않았습니다. 모델 정의 셀을 다시 실행하세요.")

✅ 모델 로드 완료! (클래스 수: 75, 장치: cuda)


In [17]:
############################################################
# 5. 모델 정의 (Faster R-CNN + EfficientNet-B3 전이학습)
############################################################

# [수정] 옵튜나가 찾은 최적의 파라미터를 변수에 할당 (찾지 못했을 경우를 대비해 기본값 설정)
best_params = getattr(study, 'best_params', {
    'lr_backbone': 1e-4,
    'lr_head': 5e-4,
    'weight_decay': 1e-4
})

# 사전학습된 모델 로드
model = get_model(num_classes)
model.to(DEVICE)

############################################################
# 6. 학습 루프 (EfficientNet-B3용 권장 버전)
############################################################

num_epochs = 15  # 기존 5보다 늘리는 걸 추천

# backbone / head 파라미터 분리 (도영님 로직 반영)
backbone_params = []
head_params = []

for name, p in model.named_parameters():
    if not p.requires_grad:
        continue
    if name.startswith("backbone"):
        backbone_params.append(p)
    else:
        head_params.append(p)

# [수정] 옵티마이저 설정 시 옵튜나에서 찾은 최적의 lr과 weight_decay 적용
optimizer = optim.AdamW(
    [
        {"params": backbone_params, "lr": best_params['lr_backbone']}, # 옵튜나 값 자동 입력
        {"params": head_params, "lr": best_params['lr_head']},         # 옵튜나 값 자동 입력
    ],
    weight_decay=best_params['weight_decay'] # 옵튜나 값 자동 입력
)

# 스케줄러 설정
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.1)

# [수정] AMP (torch.amp.GradScaler 사용)
scaler = torch.amp.GradScaler('cuda', enabled=(DEVICE.type == "cuda"))

# 작은 batch에서 EfficientNet의 BN 흔들림 방지 함수
def set_bn_eval(module):
    if isinstance(module, (nn.BatchNorm1d, nn.BatchNorm2d, nn.BatchNorm3d, nn.SyncBatchNorm)):
        module.eval()

best_val_loss = float("inf")

for epoch in range(num_epochs):
    ########################################################
    # 1) Train
    ########################################################
    model.train()

    # backbone 안의 BatchNorm은 고정 (전이학습 성능 유지용)
    model.backbone.apply(set_bn_eval)

    train_loss_sum = 0.0

    for images, targets in train_loader:
        images = [img.to(DEVICE) for img in images]
        targets = [{k: v.to(DEVICE) for k, v in t.items()} for t in targets]

        optimizer.zero_grad(set_to_none=True)

        # [수정] 최신 autocast 문법 적용
        with torch.amp.autocast('cuda', enabled=(DEVICE.type == "cuda")):
            loss_dict = model(images, targets)
            losses = sum(loss for loss in loss_dict.values())

        scaler.scale(losses).backward()

        # gradient exploding 방지용
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)

        scaler.step(optimizer)
        scaler.update()

        train_loss_sum += losses.item()

    avg_train_loss = train_loss_sum / max(1, len(train_loader))

    ########################################################
    # 2) Validation loss
    ########################################################
    # torchvision detection 모델은 loss를 얻으려면 train 모드로 호출해야 함
    model.train()
    model.backbone.apply(set_bn_eval)

    val_loss_sum = 0.0
    with torch.no_grad():
        for images, targets in val_loader:
            images = [img.to(DEVICE) for img in images]
            targets = [{k: v.to(DEVICE) for k, v in t.items()} for t in targets]

            with torch.amp.autocast('cuda', enabled=(DEVICE.type == "cuda")):
                loss_dict = model(images, targets)
                losses = sum(loss for loss in loss_dict.values())

            val_loss_sum += losses.item()

    avg_val_loss = val_loss_sum / max(1, len(val_loader))

    print(
        f"[Epoch {epoch+1}/{num_epochs}] "
        f"train_loss: {avg_train_loss:.4f} | "
        f"val_loss: {avg_val_loss:.4f}"
    )

    scheduler.step()

    # best 모델 저장
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(model.state_dict(), "best_fasterrcnn_efficientnet_b3.pth")
        print("✅ best 모델 저장 완료")

# 마지막 모델도 저장
torch.save(model.state_dict(), "last_fasterrcnn_efficientnet_b3.pth")
print("✅ 최종 모델 저장 완료")

[Epoch 1/15] train_loss: 0.4395 | val_loss: 0.2988
✅ best 모델 저장 완료
[Epoch 2/15] train_loss: 0.2350 | val_loss: 0.2312
✅ best 모델 저장 완료
[Epoch 3/15] train_loss: 0.1647 | val_loss: 0.2026
✅ best 모델 저장 완료
[Epoch 4/15] train_loss: 0.1501 | val_loss: 0.1884
✅ best 모델 저장 완료
[Epoch 5/15] train_loss: 0.1423 | val_loss: 0.1581
✅ best 모델 저장 완료
[Epoch 6/15] train_loss: 0.0906 | val_loss: 0.1021
✅ best 모델 저장 완료
[Epoch 7/15] train_loss: 0.0777 | val_loss: 0.0974
✅ best 모델 저장 완료
[Epoch 8/15] train_loss: 0.0724 | val_loss: 0.0931
✅ best 모델 저장 완료
[Epoch 9/15] train_loss: 0.0703 | val_loss: 0.0902
✅ best 모델 저장 완료
[Epoch 10/15] train_loss: 0.0674 | val_loss: 0.0889
✅ best 모델 저장 완료
[Epoch 11/15] train_loss: 0.0623 | val_loss: 0.0856
✅ best 모델 저장 완료
[Epoch 12/15] train_loss: 0.0608 | val_loss: 0.0815
✅ best 모델 저장 완료
[Epoch 13/15] train_loss: 0.0601 | val_loss: 0.0826
[Epoch 14/15] train_loss: 0.0600 | val_loss: 0.0835
[Epoch 15/15] train_loss: 0.0585 | val_loss: 0.0812
✅ best 모델 저장 완료
✅ 최종 모델 저장 완료


In [25]:
############################################################
# 7. test_images에 대해 예측 → submission.csv 생성
############################################################

# test 이미지 파일 목록 가져오기
test_files = [f for f in os.listdir(TEST_IMG_DIR) if f.endswith(".png")]
test_files = sorted(test_files)

# 설정
score_threshold = 0.3
target_size = 640  # 학습 시 설정한 크기
rows = []
model.eval()


# test_files는 ['K-001900...', '...'] 식의 실제 파일명 리스트여야 합니다.
with torch.no_grad():
    for f in test_files: # f는 "K-001900...png" 형태
        img_path = os.path.join(TEST_IMG_DIR, f)
        image = Image.open(img_path).convert("RGB")
        orig_w, orig_h = image.size

        # [중요] image_id를 파일명(확장자 포함) 그대로 사용하거나
        # 정답지 형식에 맞춰서 넣어줘야 합니다.
        image_id = f # 파일명 전체 ("K-001900...png")

        img_input = image.resize((640, 640))
        img_tensor = T.ToTensor()(img_input).to(DEVICE)
        outputs = model([img_tensor])[0]

        scores = outputs["scores"].cpu()
        keep = scores >= score_threshold
        boxes = outputs["boxes"].cpu()[keep]
        labels = outputs["labels"].cpu()[keep]
        scores = scores[keep]

        for box, lab, sc in zip(boxes, labels, scores):
            x1, y1, x2, y2 = box.tolist()
            x1 *= (orig_w / 640); y1 *= (orig_h / 640)
            x2 *= (orig_w / 640); y2 *= (orig_h / 640)

            rows.append({
                "image_id": image_id, # 여기서 파일명이 들어가야 함!
                "category_id": model2orig[int(lab.item()) - 1],
                "bbox_x": x1, "bbox_y": y1, "bbox_w": x2-x1, "bbox_h": y2-y1,
                "score": float(sc.item()),
            })

# DataFrame 생성
df_sub = pd.DataFrame(rows)

if not df_sub.empty:
    # 이미지별 상위 4개 추출
    df_sub = df_sub.sort_values(by=["image_id", "score"], ascending=[True, False])
    df_sub = df_sub.groupby("image_id").head(4)

    # annotation_id 순차 부여
    df_sub.insert(0, "annotation_id", range(1, len(df_sub) + 1))

# CSV 저장
output_path = "final_submission_fixed.csv"
df_sub.to_csv(output_path, index=False)

print(f"✅ 수정된 좌표로 생성 완료: {output_path}")
print(f"📊 총 예측 객체 수: {len(df_sub)}")

✅ 수정된 좌표로 생성 완료: final_submission_fixed.csv
📊 총 예측 객체 수: 3238


In [28]:
print(test_files[:3])

['1.png', '10.png', '100.png']


In [30]:
############################################################
# 8. 모델 성능 평가 (mAP 측정)
############################################################

import json
import os
import re
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

# 1. 정답지 로드
coco_gt = COCO(TEST_JSON_PATH)

eval_results = []
match_count = 0

for _, row in df_sub.iterrows():
    # '1.png' 같은 문자열에서 숫자 '1'만 추출합니다.
    img_name = str(row["image_id"])
    match = re.search(r'\d+', img_name)

    if match:
        actual_id = int(match.group()) # "1.png" -> 1

        # 정답지에 이 ID가 존재하는지 확인
        if actual_id in coco_gt.imgs:
            eval_results.append({
                "image_id": actual_id,
                "category_id": int(row["category_id"]),
                "bbox": [
                    float(row["bbox_x"]),
                    float(row["bbox_y"]),
                    float(row["bbox_w"]),
                    float(row["bbox_h"])
                ],
                "score": float(row["score"])
            })
            match_count += 1

print(f"✅ 매칭 성공 개수: {match_count} / {len(df_sub)}")

# 2. 평가 실행
if match_count == 0:
    print("❌ 여전히 매칭 실패! JSON의 이미지 ID와 파일 숫자명이 맞지 않습니다.")
else:
    with open("final_numeric_fix.json", "w") as f:
        json.dump(eval_results, f)

    coco_dt = coco_gt.loadRes("final_numeric_fix.json")
    coco_eval = COCOeval(coco_gt, coco_dt, "bbox")

    # 평가 실행
    coco_eval.evaluate()
    coco_eval.accumulate()
    coco_eval.summarize()

    print(f"\n🏆 최종 mAP (IoU 0.50:0.95): {coco_eval.stats[0]:.4f}")

loading annotations into memory...
Done (t=0.06s)
creating index...
index created!
✅ 매칭 성공 개수: 3238 / 3238
Loading and preparing results...
DONE (t=0.28s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=1.47s).
Accumulating evaluation results...
DONE (t=0.44s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.352
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.382
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.379
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = -1.000
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = -1.000
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.352
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.932
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.932
 Average Recall     (AR) @[ IoU=0.50:0.

## 💊 **Pill Detection Project: 최종 학습 리포트**

---

### 1. 📊 모델 성능 요약 (Final Score)
모델이 약(Pill) 객체를 탐지하는 데 있어 매우 안정적인 검출력을 보여주었습니다.

| 지표 (Metric) | 점수 (Value) | 의미 분석 |
| :--- | :--- | :--- |
| **mAP (IoU 0.50:95)** | **0.3523** | 전체적인 탐지 정확도 (Base 모델 대비 **우수**) |
| **mAP@0.50** | **0.3820** | 박스가 정답과 50%만 겹쳐도 정답 처리 시 점수 |
| **Average Recall (AR)** | **0.9320** | **핵심 성과!** 전체 정답 중 **93.2%를 찾아냄** |

> **종합 평가:** 모델이 약의 위치를 놓치는 경우는 거의 없으나(Recall 매우 높음), 박스의 정밀한 위치(Localization)를 조금 더 정교하게 다듬으면 0.4 이상의 고득점도 충분히 가능한 상태입니다.

---

### 2. 🛠️ 모델 구성 및 학습 설정
* **Model:** Faster R-CNN (Backbone: EfficientNet-B3)
* **Input Size:** 640 x 640
* **Epochs:** 15
* **Device:** CUDA (GPU)

---

### 3. ✅ 주요 해결 과제
1. **Background Class:** 카테고리 ID를 1부터 시작하도록 조정하여 배경 오류 해결.
2. **Coordinate Scaling:** 640 좌표를 원본 이미지 크기로 복구하는 로직 적용.
3. **ID Matching:** 파일명(`1.png`)과 정답지 ID(`1`)를 정규표현식으로 매칭 성공.